In [0]:
%run ../config/set_up_env_paths

In [0]:
import base64
import requests
import json
from config.core import config

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS nbp.gold.export 
COMMENT "Gold table export volume"

In [0]:

github_token : str = dbutils.secrets.get(scope="Life_ETL", key="Github_token")
repo : str = config.github.repo
owner : str= config.github.owner
target_path : str = "data/gold_table.csv" 
url :str = f"https://api.github.com/repos/{owner}/Life_ETL/contents/{target_path}"
temp_path : str  = f"/Volumes/{config.catalog.catalog_name}/{config.catalog.gold_schema}/export/{config.catalog.gold_table}"
headers = {
    "Authorization": f"token {github_token}",
    "Accept": "application/vnd.github+json"
}


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-5991752458382722>, line 2
      1 github_token : str = dbutils.secrets.get(scope="Life_ETL", key="Github_token")
----> 2 repo : str = config.github.repo
      3 owner : str= config.github.owner
      4 temp_path : str  = "/Volumes/nbp/gold/exports/export_test"

File /databricks/python/lib/python3.12/site-packages/pydantic/main.py:891, in BaseModel.__getattr__(self, item)
    888     return super().__getattribute__(item)  # Raises AttributeError if appropriate
    889 else:
    890     # this is the current error
--> 891     raise AttributeError(f'{type(self).__name__!r} object has no attribute {item!r}')

AttributeError: 'Config' object has no attribute 'github'

In [0]:
gold_df = spark.table(f"{config.catalog.catalog_name}.{config.catalog.gold_schema}.{config.catalog.gold_table}")

gold_df.coalesce(1).write.mode("overwrite").option("header", True).csv(temp_path)

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-5991752458382729>, line 1
----> 1 gold_df = spark.table(f"{config.catalog.catalog_name}.{config.catalog.gold_schema}.{config.catalog.gold_table}")

File /databricks/python/lib/python3.12/site-packages/pydantic/main.py:891, in BaseModel.__getattr__(self, item)
    888     return super().__getattribute__(item)  # Raises AttributeError if appropriate
    889 else:
    890     # this is the current error
--> 891     raise AttributeError(f'{type(self).__name__!r} object has no attribute {item!r}')

AttributeError: 'UCConfig' object has no attribute 'gold_table'

In [0]:
files = dbutils.fs.ls(temp_path)

[FileInfo(path='dbfs:/Volumes/nbp/gold/export/test_table/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1764006177000),
 FileInfo(path='dbfs:/Volumes/nbp/gold/export/test_table/_committed_1438521960764156787', name='_committed_1438521960764156787', size=113, modificationTime=1764006176000),
 FileInfo(path='dbfs:/Volumes/nbp/gold/export/test_table/_started_1438521960764156787', name='_started_1438521960764156787', size=0, modificationTime=1764006176000),
 FileInfo(path='dbfs:/Volumes/nbp/gold/export/test_table/part-00000-tid-1438521960764156787-3480ec7a-0358-4028-a312-152ee88bfd98-170-1-c000.csv', name='part-00000-tid-1438521960764156787-3480ec7a-0358-4028-a312-152ee88bfd98-170-1-c000.csv', size=58514, modificationTime=1764006176000)]

In [0]:
table_csv_file= [f for f in files if f.path.endswith("csv")]
table_csv_path = table_csv_file[0].path
correct_file_path = table_csv_path.split(":")[1]

dbfs:/Volumes/nbp/gold/export/test_table/part-00000-tid-1438521960764156787-3480ec7a-0358-4028-a312-152ee88bfd98-170-1-c000.csv


In [0]:
with open(correct_file_path, "r") as f:
  csv_content = f.read()
content_b64 = base64.b64encode(csv_content.encode()).decode()

try:
    r_get = requests.get(url, headers=headers)
    if r_get.status_code == 200:
        sha = r_get.json()["sha"] # if file exists , get sha
        print(f"File alredy exists, fetch SHA from GitHub {sha}")
    elif r_get.status_code == 404:
        print("File does not exist - create new file")
        sha = None # New file
    else:
        r_get.raise_for_status()
except requests.exceptions.HTTPError as err:
    print(f"HTTP error occurred: {err}")
    raise Exception(f"Stopping pipeline due to the error in get method with GitHub - {err}")

payload = {
    "message": "Automated upload from Databricks",
    "content": content_b64,
    "branch": "main",
}
if sha:
    payload["sha"] = sha

try:
    r_put = requests.put(url, headers=headers, data=json.dumps(payload))
    r_put.raise_for_status()
except requests.exceptions.HTTPError as err:
    print("HTTP error occurred:", err)
    raise Exception(f"Stopping pipeline due to the error in put method with GitHub - {err}") 


Updating the file SHA 58d04db2910d077b2ddc96e576f1a61299920fff


In [0]:
# Delete files from DBFS
for file in files:
  dbutils.fs.rm(file.path, recurse=True)